In [ ]:
%run ./00_load_and_preprocess.ipynb

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

print("Dataset shape:", pax.shape)
print("Target:", TARGET)

In [ ]:
# Route identifiers
ROUTE_COLS = ["ORG_AIRPRT", "DES_AIRPRT"]
ROUTE_MONTH_COLS = ROUTE_COLS + ["DEP_MONTH"]

# Original features
BASE_FEATURES = [
    "DEP_YEAR",
    "DEP_MONTH",
    "ORG_AIRPRT",
    "DES_AIRPRT"
]

# Historical features
HIST_FEATURES = [
    "ROUTE_HIST_MEAN",
    "ROUTE_HIST_COUNT",
    "ROUTE_MONTH_HIST_MEAN"
]

FEATURES_HIST = BASE_FEATURES + HIST_FEATURES

# Features used by the preprocessing pipeline
NUMERIC_FEATURES = [
    "DEP_YEAR",
    "DEP_MONTH",
    "ROUTE_HIST_MEAN",
    "ROUTE_HIST_COUNT",
    "ROUTE_MONTH_HIST_MEAN"
]

CATEGORICAL_FEATURES = [
    "ORG_AIRPRT",
    "DES_AIRPRT"
]

# XGBoost configuration
XGB_PARAMS = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": 4
}

# Convert year/month to dates
row_dates = pd.to_datetime(
    dict(
        year=pax["DEP_YEAR"],
        month=pax["DEP_MONTH"],
        day=1
    )
)

print("Features:", FEATURES_HIST)
print("Date range:", row_dates.min(), "to", row_dates.max())

In [ ]:
def add_expanding_features(train):
    """
    Create historical features for training.

    Each observation uses only previous observations
    from the same route or route-month group.
    """

    train = train.sort_values(
        ["DEP_YEAR", "DEP_MONTH"]
    ).copy()

    # Historical route count and mean
    route_group = train.groupby(ROUTE_COLS)[TARGET]

    train["ROUTE_HIST_COUNT"] = route_group.cumcount()

    train["ROUTE_HIST_MEAN"] = (
        route_group.cumsum() - train[TARGET]
    ) / train["ROUTE_HIST_COUNT"].replace(0, np.nan)

    # Historical same-month route mean
    month_group = train.groupby(ROUTE_MONTH_COLS)[TARGET]

    month_count = month_group.cumcount()

    train["ROUTE_MONTH_HIST_MEAN"] = (
        month_group.cumsum() - train[TARGET]
    ) / month_count.replace(0, np.nan)

    # Fall back to overall historical route mean
    train["ROUTE_MONTH_HIST_MEAN"] = (
        train["ROUTE_MONTH_HIST_MEAN"]
        .fillna(train["ROUTE_HIST_MEAN"])
    )

    return train


def add_frozen_features(history, future):
    """
    Create validation features using only data
    available before the forecast origin.
    """

    future = future.copy()

    # Route-level historical statistics
    route_stats = (
        history.groupby(ROUTE_COLS)[TARGET]
        .agg(
            ROUTE_HIST_MEAN="mean",
            ROUTE_HIST_COUNT="count"
        )
        .reset_index()
    )

    # Same-month historical route averages
    month_stats = (
        history.groupby(ROUTE_MONTH_COLS)[TARGET]
        .mean()
        .rename("ROUTE_MONTH_HIST_MEAN")
        .reset_index()
    )

    # Attach route statistics
    future = future.merge(
        route_stats,
        on=ROUTE_COLS,
        how="left",
        validate="many_to_one"
    )

    # Attach seasonal statistics
    future = future.merge(
        month_stats,
        on=ROUTE_MONTH_COLS,
        how="left",
        validate="many_to_one"
    )

    # Fall back to overall route mean
    future["ROUTE_MONTH_HIST_MEAN"] = (
        future["ROUTE_MONTH_HIST_MEAN"]
        .fillna(future["ROUTE_HIST_MEAN"])
    )

    return future


print("Historical feature functions ready!")

In [ ]:
def make_xgb_pipeline(numeric_features, categorical_features):
    preprocessor = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(**XGB_PARAMS))
    ])

    return model


# Main model: 7 features
xgb_hist = make_xgb_pipeline(
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES
)

# Baseline model: original 4 features
xgb_base = make_xgb_pipeline(
    ["DEP_YEAR", "DEP_MONTH"],
    CATEGORICAL_FEATURES
)


def evaluate_model(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    }


print("XGBoost pipelines ready!")

In [ ]:
# Chronological split
train_mask = row_dates < pd.Timestamp("2024-01-01")

val_mask = (
    (row_dates >= pd.Timestamp("2024-01-01")) &
    (row_dates < pd.Timestamp("2026-01-01"))
)

train_24 = pax.loc[train_mask].copy()
val_24 = pax.loc[val_mask].copy()

# Feature engineering
train_24 = add_expanding_features(train_24)
val_24 = add_frozen_features(train_24, val_24)

# Model inputs
X_train_hist = train_24[FEATURES_HIST]
y_train_hist = train_24[TARGET]

X_val_hist = val_24[FEATURES_HIST]
y_val_hist = val_24[TARGET]

print("Training observations:", len(train_24))
print("Validation observations:", len(val_24))
print("Validation months:", row_dates.loc[val_mask].nunique())

print("\nMissing validation features:")
print(val_24[HIST_FEATURES].isna().sum())

assert len(val_24) == val_mask.sum()
assert row_dates.loc[val_mask].nunique() == 24
assert not val_24[HIST_FEATURES].isna().any().any()

# Train the main model
xgb_hist.fit(X_train_hist, y_train_hist)

# Predict
pred_hist = xgb_hist.predict(X_val_hist)

# Evaluate
main_metrics = evaluate_model(y_val_hist, pred_hist)

print("\n=== Historical Feature XGBoost: 2024–2025 ===")
for metric, value in main_metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
results = val_24[
    ["DEP_YEAR", "DEP_MONTH", TARGET]
].copy()

results["PREDICTED_PAX"] = pred_hist

yearly_metrics = []

for year in [2024, 2025]:
    subset = results[results["DEP_YEAR"] == year]

    metrics = evaluate_model(
        subset[TARGET],
        subset["PREDICTED_PAX"]
    )

    yearly_metrics.append({
        "Year": year,
        **metrics
    })

yearly_results = pd.DataFrame(yearly_metrics)

print("=== Year-by-Year Validation ===")
display(yearly_results.round(4))

In [ ]:
# Baseline 1: Original XGBoost
xgb_base.fit(
    train_24[BASE_FEATURES],
    train_24[TARGET]
)

pred_base = xgb_base.predict(
    val_24[BASE_FEATURES]
)

base_metrics = evaluate_model(
    y_val_hist,
    pred_base
)

# Baseline 2: Historical route mean
route_mean_pred = val_24["ROUTE_HIST_MEAN"]

route_mean_metrics = evaluate_model(
    y_val_hist,
    route_mean_pred
)

# Compare results
comparison = pd.DataFrame([
    {
        "Model": "Original XGBoost",
        **base_metrics
    },
    {
        "Model": "Historical Route Mean",
        **route_mean_metrics
    },
    {
        "Model": "Historical Feature XGBoost",
        **main_metrics
    }
])

print("=== 2024–2025 Model Comparison ===")
display(comparison.round(4))

In [ ]:
# Earlier chronological split
early_train_mask = row_dates < pd.Timestamp("2022-01-01")

early_val_mask = (
    (row_dates >= pd.Timestamp("2022-01-01")) &
    (row_dates < pd.Timestamp("2024-01-01"))
)

early_train = pax.loc[early_train_mask].copy()
early_val = pax.loc[early_val_mask].copy()

# Historical features
early_train = add_expanding_features(early_train)
early_val = add_frozen_features(early_train, early_val)

# Inputs
X_early_train = early_train[FEATURES_HIST]
y_early_train = early_train[TARGET]

X_early_val = early_val[FEATURES_HIST]
y_early_val = early_val[TARGET]

print("Early training observations:", len(early_train))
print("Early validation observations:", len(early_val))

print("\nMissing validation features:")
print(early_val[HIST_FEATURES].isna().sum())

assert len(early_val) == early_val_mask.sum()
assert row_dates.loc[early_val_mask].nunique() == 24
assert not early_val[HIST_FEATURES].isna().any().any()

# Fresh model with identical hyperparameters
xgb_early = clone(xgb_hist)

xgb_early.fit(X_early_train, y_early_train)

# Predictions
pred_early = xgb_early.predict(X_early_val)

# Evaluation
early_metrics = evaluate_model(
    y_early_val,
    pred_early
)

# Historical mean baseline
early_baseline_metrics = evaluate_model(
    y_early_val,
    early_val["ROUTE_HIST_MEAN"]
)

early_comparison = pd.DataFrame([
    {
        "Model": "Historical Route Mean",
        **early_baseline_metrics
    },
    {
        "Model": "Historical Feature XGBoost",
        **early_metrics
    }
])

print("\n=== 2022–2023 Backtest ===")
display(early_comparison.round(4))

In [ ]:
final_summary = pd.DataFrame([
    {
        "Validation Period": "2022–2023",
        "Training Period": "2019–2021",
        **early_metrics
    },
    {
        "Validation Period": "2024–2025",
        "Training Period": "2019–2023",
        **main_metrics
    }
])

print("=== XGBoost 24-Month Backtest Summary ===")
display(final_summary.round(4))

improvement = (
    (route_mean_metrics["MAE"] - main_metrics["MAE"])
    / route_mean_metrics["MAE"]
    * 100
)

print(
    f"\nMAE improvement over historical route mean: "
    f"{improvement:.2f}%"
)